# NLP Text Preprocessing Pipeline
**Libraries Explored:** NLTK, RegEx, spaCy, Contractions  
**Dataset:** Twitter Tweets Sentiment Dataset (`Tweets.csv`)

In [ ]:
pip install contractions

In [ ]:


import re
import html
import pandas as pd
import contractions
import spacy
import nltk

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk import pos_tag


for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)


nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

## 2. Load and Inspect Dataset

In [ ]:

df = pd.read_csv("/content/Tweets.csv")

print("Initial Dataset Shape:", df.shape)
print("Columns:", list(df.columns))

# Drop missing text entries
df = df.dropna(subset=["text"]).reset_index(drop=True)
print("Shape after dropping nulls:", df.shape)
df.head(5)

Initial Dataset Shape: (27481, 4)
Columns: ['textID', 'text', 'selected_text', 'sentiment']
Shape after dropping nulls: (27480, 4)


,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative


## 3. Preprocessing Steps

In [ ]:
sample_text = "I'd have responded, if I were going! Check out https://example.com #NLP @AI_dev"

# Step 1: RegEx & HTML cleaning
clean_html = html.unescape(sample_text)
clean_urls = re.sub(r"http\S+|www\.\S+", " ", clean_html)
clean_mentions = re.sub(r"@\w+", " ", clean_urls)
clean_hashtags = re.sub(r"#(\w+)", r"\1", clean_mentions)

# Step 2: Contraction Expansion
expanded = contractions.fix(clean_hashtags)

# Step 3: Punctuation & Unwanted Characters
alpha_only = re.sub(r"[^a-zA-Z\s]", " ", expanded).lower()
normalized_space = re.sub(r"\s+", " ", alpha_only).strip()

print("Original :", sample_text)
print("Cleaned  :", normalized_space)

Original : I'd have responded, if I were going! Check out https://example.com #NLP @AI_dev
Cleaned  : i would have responded if i were going check out nlp


## 4. Linguistic Processing: NLTK vs. spaCy

In [ ]:
#  NLTK Processing
def nltk_pos_to_wordnet(tag):
    if tag.startswith("J"): return wordnet.ADJ
    elif tag.startswith("V"): return wordnet.VERB
    elif tag.startswith("N"): return wordnet.NOUN
    elif tag.startswith("R"): return wordnet.ADV
    return wordnet.NOUN

nltk_tokens = word_tokenize(normalized_space)
nltk_filtered = [t for t in nltk_tokens if t not in stop_words and len(t) > 2]
nltk_tags = pos_tag(nltk_filtered)
nltk_lemmas = [lemmatizer.lemmatize(w, nltk_pos_to_wordnet(t)) for w, t in nltk_tags]
nltk_stems = [stemmer.stem(w) for w in nltk_filtered]

#  spaCy Processing
spacy_doc = nlp(normalized_space)
spacy_lemmas = [token.lemma_ for token in spacy_doc if not token.is_stop and len(token.text) > 2]

print("Filtered Tokens :", nltk_filtered)
print("NLTK Lemmatized :", nltk_lemmas)
print("spaCy Lemmatized:", spacy_lemmas)
print("NLTK Stemmed    :", nltk_stems)

Filtered Tokens : ['would', 'responded', 'going', 'check', 'nlp']
NLTK Lemmatized : ['would', 'respond', 'go', 'check', 'nlp']
spaCy Lemmatized: ['respond', 'go', 'check', 'nlp']
NLTK Stemmed    : ['would', 'respond', 'go', 'check', 'nlp']


## 5. End-to-End Pipeline Application

In [ ]:
def preprocess_pipeline(text):
    # 1. RegEx + HTML
    text = html.unescape(str(text))
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\brt\b", " ", text, flags=re.I)

    # 2. Contractions
    text = contractions.fix(text)

    # 3. Clean characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text).lower()
    text = re.sub(r"\s+", " ", text).strip()

    # 4. Tokenize & Stop words
    tokens = [w for w in word_tokenize(text) if w not in stop_words and len(w) > 2]

    # 5. Lemmatization & Stemming
    tags = pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(w, nltk_pos_to_wordnet(t)) for w, t in tags]
    stems = [stemmer.stem(w) for w in tokens]

    return pd.Series({
        "cleaned_text": text,
        "tokens": tokens,
        "lemmatized": " ".join(lemmas),
        "stemmed": " ".join(stems)
    })

# Process the full dataset
processed_df = df["text"].apply(preprocess_pipeline)
df_final = pd.concat([df, processed_df], axis=1)

# Remove empty results after preprocessing
df_final = df_final[df_final["tokens"].apply(len) > 0].reset_index(drop=True)
print("Final Dataset Shape:", df_final.shape)

Final Dataset Shape: (27362, 8)


## 6. Preprocessing Results & Comparisons

In [ ]:

comparison_cols = ["text", "cleaned_text", "lemmatized", "stemmed", "sentiment"]
df_final[comparison_cols].head(5)

,text,cleaned_text,lemmatized,stemmed,sentiment
0,"I`d have responded, if I were going",i d have responded if i were going,respond go,respond go,neutral
1,Sooo SAD I will miss you here in San Diego!!!,sooo sad i will miss you here in san diego,sooo sad miss san diego,sooo sad miss san diego,negative
2,my boss is bullying me...,my boss is bullying me,bos bully,boss bulli,negative
3,what interview! leave me alone,what interview leave me alone,interview leave alone,interview leav alon,negative
4,"Sons of ****, why couldn`t they put them on t...",sons of why couldn t they put them on the rele...,son put release already buy,son put releas alreadi bought,negative


In [ ]:

df_final.to_csv("tweets_preprocessed.csv", index=False)
print("Saved preprocessed dataset to 'tweets_preprocessed.csv'")

Saved preprocessed dataset to 'tweets_preprocessed.csv'
